# Intégration de modèles IA avec Python

## Travailler avec des modèles IA gratuits : Google Generative AI, Ollama et OpenRouter

Ce notebook explique comment intégrer différents modèles d'intelligence artificielle avec Python, **sans payer et sans carte bancaire**. Nous explorerons trois solutions gratuites : **Google Generative AI** (modèles Gemini), **Ollama** (modèles locaux) et **OpenRouter** (modèles `:free`). Nous verrons aussi **Groq**, une solution bonus très rapide.

## Pourquoi intégrer l'IA avec Python ?

- **Automatisation** : réaliser des tâches répétitives ou complexes (résumés, rapports, classification)
- **Analyse de données** : extraire des informations utiles à partir de grandes quantités de texte
- **Génération de contenu** : créer du texte, des emails, des documents, du code
- **Personnalisation** : adapter les applications aux besoins de chaque utilisateur
- **Apprentissage** : comprendre comment les modèles de langage fonctionnent en les manipulant

Tous les exemples de ce notebook utilisent des modèles accessibles **gratuitement** (avec ou sans inscription gratuite).

## Qu'est-ce qu'un modèle de langage (LLM) ?

Un **LLM** (Large Language Model) est un modèle d'IA entraîné sur d'énormes quantités de texte. Il prédit le mot suivant le plus probable, ce qui lui permet de répondre à des questions, de résumer, de traduire, etc.

### Notions importantes

- **Token** : unité de texte (environ 3/4 d'un mot en anglais). Les API comptent les tokens pour leurs limites d'utilisation
- **Prompt** : le message que vous envoyez au modèle
- **Contexte** : la mémoire du modèle (fenêtre de tokens qu'il peut traiter)
- **Température** : le niveau de créativité (0 = très rigoureux, 1 = très créatif)

### Les 3 façons d'utiliser un modèle avec Python

1. **API cloud** : le modèle tourne sur des serveurs distants (Google, OpenRouter, Groq)
2. **Modèle local** : le modèle est téléchargé et tourne sur votre machine (Ollama)
3. **Bibliothèques embarquées** : pas d'API ni de serveur, mais souvent plus lourdes (transformers, scikit-learn)

## Les fournisseurs gratuits présentés dans ce notebook

| Fournisseur | Type | Coût | Particularité |
|-------------|------|------|---------------|
| **Google Generative AI** | Cloud | Gratuit (free tier) | Modèles Gemini, pas de carte bancaire, clé API sur AI Studio |
| **Ollama** | Local | 100 % gratuit | Hors ligne, idéal pour les ordinateurs modestes |
| **OpenRouter** | Cloud | Gratuit (modèles `:free`) | Un seul compte pour des dizaines de modèles |
| **Groq** (bonus) | Cloud | Gratuit (free tier) | Très rapide (puces LPU), modèles opensource |

> **Point important** : aucune de ces solutions ne demande de carte bancaire pour commencer. Les limites gratuites suffisent largement pour apprendre et prototyper.

## Prix des différents modèles (2026)

| Fournisseur | Modèle | Type | Prix gratuit | Prix payant (1M tokens) |
|-------------|--------|------|--------------|--------------------------|
| Google AI | Gemini 2.5 Flash | Cloud | Gratuit (10 req/min) | 0,30 $ entrée / 2,50 $ sortie |
| Google AI | Gemini 2.5 Flash-Lite | Cloud | Gratuit (15 req/min) | ≈ 0,10 $ / 0,40 $ |
| Google AI | Gemini 2.5 Pro | Cloud | Plus dans le free tier (2026) | 1,25 $ / 10 $ |
| Ollama | Qwen 2.5 0.5B / 1.5B / 3B | Local | Gratuit, hors ligne | — |
| Ollama | Llama 3.2 1B / 3B | Local | Gratuit, hors ligne | — |
| Ollama | DeepSeek-R1 1.5B | Local | Gratuit, hors ligne | — |
| OpenRouter | Modèles `:free` (ex: `openrouter/free`) | Cloud | Gratuit (20 req/min, 50 req/jour) | Payant selon le modèle choisi |
| Groq | Llama 3.1 8B Instant | Cloud | Gratuit (30 req/min, 14 400 req/jour) | 0,05 $ / 0,08 $ |

> ⚠️ **Les prix et les limites changent régulièrement.** Vérifiez toujours les tarifs officiels avant de vous lancer dans un projet. En 2026, les modèles **Flash** de Google sont les seuls encore gratuits (les Pro sont devenus payants).

## Architecture d'intégration

Pour gérer plusieurs fournisseurs sans dupliquer de code, nous allons utiliser une approche **orientée objet** :

1. **Classe de base `AIProvider`** : interface commune à tous les fournisseurs
2. **Implémentations spécifiques** : une classe par fournisseur (Google, Ollama, OpenRouter, Groq)
3. **Gestion des prompts** : formatage et envoi des requêtes
4. **Récupération des réponses** : traitement des résultats

Cette approche permet de **changer de fournisseur sans modifier le reste du programme** : il suffit de remplacer l'objet.

In [ ]:
# Installation des bibliothèques requises
!pip install google-genai ollama openai groq

## Configuration des clés API

### Où créer les clés gratuites ?

- **Google (Gemini)** : [aistudio.google.com/apikey](https://aistudio.google.com/apikey) → clé `GEMINI_API_KEY`, gratuite, sans carte bancaire
- **OpenRouter** : [openrouter.ai/keys](https://openrouter.ai/keys) → clé `OPENROUTER_API_KEY`, gratuite, sans carte bancaire
- **Groq** : [console.groq.com/keys](https://console.groq.com/keys) → clé `GROQ_API_KEY`, gratuite, sans carte bancaire

### Deux méthodes pour stocker les clés

1. **Variables d'environnement** : `export GEMINI_API_KEY="...."` dans le terminal (ou un fichier `.env`)
2. **Saisie directe** : le notebook demande la clé (elle ne sera pas visible à l'écran)

> ⚠️ **Ne partagez jamais vos clés API** et ne les écrivez pas en dur dans le code : elles donnent accès à votre compte.

In [ ]:
import os
import getpass

# Récupération des clés depuis les variables d'environnement
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

# Si une clé Google manque, on la demande à l'utilisateur
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("Entrez votre clé API Google (Gemini) : ")

print(f"Clé Google : {'✔ présente' if GEMINI_API_KEY else '✘ absente'}")
print(f"Clé OpenRouter : {'✔ présente' if OPENROUTER_API_KEY else '✘ absente'}")
print(f"Clé Groq : {'✔ présente' if GROQ_API_KEY else '✘ absente'}")

## Classe de base pour l'IA

Créons une classe abstraite qui définit **l'interface commune** à tous les fournisseurs :

In [ ]:
from abc import ABC, abstractmethod

class AIProvider(ABC):
    """
    Classe abstraite : interface commune aux différents fournisseurs d'IA.
    Tous les fournisseurs (Google, Ollama, OpenRouter, Groq) hériteront de cette classe.
    """

    @abstractmethod
    def generate_text(self, prompt: str) -> str:
        """Génère du texte à partir d'un prompt donné."""
        pass

    @abstractmethod
    def get_model_info(self) -> dict:
        """Retourne les informations sur le fournisseur et le modèle utilisé."""
        pass

## Fournisseur 1 : Google Generative AI

Google propose les modèles **Gemini** via la bibliothèque officielle `google-genai` (le SDK Google Gen AI, qui remplace l'ancien `google-generativeai`).

### Étapes pour commencer (gratuit, sans carte bancaire)

1. Créer un compte Google (ou en avoir un)
2. Aller sur [aistudio.google.com](https://aistudio.google.com) et cliquer sur « Get API key »
3. Copier la clé et la stocker dans la variable d'environnement `GEMINI_API_KEY`

### Modèles gratuits en 2026 (free tier)

- **Gemini 2.5 Flash** : bon équilibre qualité/vitesse (10 requêtes/min)
- **Gemini 2.5 Flash-Lite** : le plus léger, plus de requêtes autorisées (15 req/min)
- **Gemma** (modèles ouverts de Google) : disponibles aussi

Les modèles **Pro** ne sont plus gratuits depuis 2026. Le free tier suffit largement pour un cours ou un prototype.

In [ ]:
from google import genai

class GoogleAIProvider(AIProvider):
    def __init__(self, api_key: str, model_name: str = "gemini-2.5-flash"):
        self.api_key = api_key
        self.model_name = model_name
        self.client = genai.Client(api_key=api_key)

    def generate_text(self, prompt: str) -> str:
        try:
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=prompt
            )
            return response.text
        except Exception as e:
            return f"Erreur lors de la génération : {str(e)}"

    def get_model_info(self) -> dict:
        return {
            "provider": "Google Generative AI",
            "model": self.model_name,
            "type": "Cloud (Free tier)"
        }

    @staticmethod
    def list_available_models(api_key: str) -> list:
        """Liste les modèles Gemini disponibles pour la clé API."""
        try:
            client = genai.Client(api_key=api_key)
            return [m.name for m in client.models.list()]
        except Exception as e:
            return [f"Erreur : {e}"]

In [ ]:
# Test du fournisseur Google Generative AI
if GEMINI_API_KEY:
    google_provider = GoogleAIProvider(GEMINI_API_KEY, model_name="gemini-2.5-flash")
    print(google_provider.get_model_info())
    print("\n→ Réponse :")
    print(google_provider.generate_text("Explique en deux phrases ce qu'est un LLM."))
else:
    print("Pas de clé API Google détectée.")
    print("Créez-en une gratuitement : https://aistudio.google.com/apikey")
    print("Puis relancez la cellule de configuration.")

## Fournisseur 2 : Ollama (modèles locaux)

[Ollama](https://ollama.com) permet de télécharger et d'exécuter des modèles **directement sur votre ordinateur**, sans internet et sans compte.

### Installation

- **Windows / macOS** : télécharger l'installateur sur [ollama.com](https://ollama.com/download)
- **Linux** : `curl -fsSL https://ollama.com/install.sh | sh`
- Le serveur local écoute sur `http://localhost:11434`

### Choisir un petit modèle pour du matériel modeste

| Modèle | Taille du fichier | Mémoire nécessaire | Idéal pour |
|--------|-------------------|--------------------|------------|
| `qwen2.5:0.5b` | ~0,4 GB | ~2 GB RAM | Tests rapides |
| `qwen2.5:1.5b` | ~1,1 GB | ~3 GB RAM | Tâches simples |
| `llama3.2:1b` | ~1,3 GB | ~2 GB RAM | Chat rapide |
| `deepseek-r1:1.5b` | ~1,1 GB | ~3 GB RAM | Raisonnement |
| `llama3.2:3b` | ~2,0 GB | ~4 GB RAM | Bon compromis qualité |
| `gemma3:4b` | ~3,3 GB | ~5 GB RAM | Meilleure qualité sur 16 GB RAM |

> 💡 **Conseil matériel** : avec moins de 8 GB de RAM, préférez les modèles de 0,5 à 3 milliards de paramètres. Un petit modèle qui répond en 2 secondes vaut mieux qu'un gros modèle qui rame.

In [ ]:
# Téléchargement d'un petit modèle (environ 1,1 GB)
!ollama pull qwen2.5:1.5b

In [ ]:
import ollama


class OllamaProvider(AIProvider):
    def __init__(
        self,
        model_name: str = "llama3.1:8b",
        host: str = "http://host.docker.internal:11434",
    ):
        self.model_name = model_name
        self.client = ollama.Client(host=host)

    def generate_text(self, prompt: str) -> str:
        try:
            response = self.client.generate(
                model=self.model_name,
                prompt=prompt,
            )
            return response["response"]
        except Exception as e:
            return f"Erreur lors de la génération : {str(e)}"

    def get_model_info(self) -> dict:
        return {
            "provider": "Ollama",
            "model": self.model_name,
            "type": "Ollama sur le serveur hôte",
            "host": self.client._client.base_url,
        }

    def list_local_models(self) -> list:
        """Liste les modèles disponibles sur l'Ollama du serveur hôte."""
        try:
            response = self.client.list()
            return [m["model"] for m in response["models"]]
        except Exception:
            return []

In [ ]:
# Test du fournisseur Ollama
ollama_provider = OllamaProvider("qwen2.5:1.5b")
print(ollama_provider.get_model_info())
print(f"\nModèles locaux disponibles : {ollama_provider.list_local_models()}")
print("\n→ Réponse :")
print(ollama_provider.generate_text("Quelle est la capitale du Japon ? Réponds en une phrase."))

## Fournisseur 3 : OpenRouter (modèles gratuits)

[OpenRouter](https://openrouter.ai) est un **routeur d'API** : un seul compte et une seule clé pour accéder à des dizaines de modèles différents (Google, Meta, Mistral, Qwen, NVIDIA...).

### Les modèles gratuits `:free`

- Les modèles gratuits portent le suffixe **`:free`** (ex : `openai/gpt-oss-20b:free`)
- Le modèle **`openrouter/free`** choisit automatiquement un modèle gratuit adapté à votre requête (recommandé pour débuter)
- **Limites** : 20 requêtes/min et 50 requêtes/jour avec un compte gratuit (aucune carte bancaire)

### Avantages

- Testez plusieurs modèles sans changer de clé API
- API compatible OpenAI : presque le même code que Groq fonctionne sans modification

> ⚠️ La liste des modèles `:free` change régulièrement. Consultez [openrouter.ai/models](https://openrouter.ai/models) (filtre « Free ») avant de dépendre d'un modèle précis.

In [ ]:
from openai import OpenAI

class OpenRouterProvider(AIProvider):
    def __init__(self, api_key: str, model_name: str = "openrouter/free"):
        self.api_key = api_key
        self.model_name = model_name
        # OpenRouter utilise une API compatible OpenAI
        self.client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

    def generate_text(self, prompt: str) -> str:
        try:
            completion = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=500
            )
            return completion.choices[0].message.content
        except Exception as e:
            return f"Erreur lors de la génération : {str(e)}"

    def get_model_info(self) -> dict:
        return {
            "provider": "OpenRouter",
            "model": self.model_name,
            "type": "Cloud (Free tier)"
        }

In [ ]:
# Test du fournisseur OpenRouter
if OPENROUTER_API_KEY:
    # "openrouter/free" choisit automatiquement un modèle gratuit
    openrouter_provider = OpenRouterProvider(OPENROUTER_API_KEY, model_name="openrouter/free")
    print(openrouter_provider.get_model_info())
    print("\n→ Réponse :")
    print(openrouter_provider.generate_text("Explique la différence entre une base de données SQL et NoSQL en 3 phrases."))
else:
    print("Pas de clé API OpenRouter détectée.")
    print("Créez-en une gratuitement : https://openrouter.ai/keys")

## Bonus : Groq (l'IA la plus rapide du marché)

[Groq](https://console.groq.com) propose une API **gratuite et sans carte bancaire** avec des modèles opensource (Llama, Qwen, DeepSeek...).

- **Spécialité** : des puces LPU ultra-rapides (des centaines de tokens par seconde)
- **Limites gratuites** : 30 requêtes/min, 14 400 requêtes/jour
- **API compatible OpenAI** : `https://api.groq.com/openai/v1`

### Modèles populaires gratuits

| Modèle | Taille | Usage |
|--------|--------|-------|
| `llama-3.1-8b-instant` | 8B | Tâches générales, très rapide |
| `llama-3.3-70b-versatile` | 70B | Meilleure qualité |
| `deepseek-r1-distill-llama-70b` | 70B | Raisonnement, mathématiques |

> 💡 Idéal pour les applications en temps réel (chat, assistants vocaux) grâce à sa très faible latence.

In [ ]:
from groq import Groq

class GroqProvider(AIProvider):
    def __init__(self, api_key: str, model_name: str = "llama-3.1-8b-instant"):
        self.api_key = api_key
        self.model_name = model_name
        self.client = Groq(api_key=api_key)

    def generate_text(self, prompt: str) -> str:
        try:
            completion = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=500
            )
            return completion.choices[0].message.content
        except Exception as e:
            return f"Erreur lors de la génération : {str(e)}"

    def get_model_info(self) -> dict:
        return {
            "provider": "Groq",
            "model": self.model_name,
            "type": "Cloud (Free tier, ultra rapide)"
        }

In [ ]:
# Test du fournisseur Groq
if GROQ_API_KEY:
    groq_provider = GroqProvider(GROQ_API_KEY, model_name="llama-3.1-8b-instant")
    print(groq_provider.get_model_info())
    print("\n→ Réponse :")
    print(groq_provider.generate_text("Donne trois idées de mini-projets Python pour débutants."))
else:
    print("Pas de clé API Groq détectée.")
    print("Créez-en une gratuitement : https://console.groq.com/keys")

## Exemple concret : génération de rapports

Grâce à l'interface commune (`AIProvider`), nous pouvons écrire des outils **indépendants du fournisseur**. Cet exemple fonctionne avec n'importe lequel des 4 fournisseurs, sans modifier le code :

In [ ]:
class ReportGenerator:
    """Génère des rapports et des résumés à partir d'un fournisseur d'IA quelconque."""

    def __init__(self, provider: AIProvider):
        self.provider = provider

    def generate_report(self, data: dict) -> str:
        prompt = "Créez un rapport d'analyse clair et professionnel basé sur les données suivantes :"
        for key, value in data.items():
            prompt += f"\n- {key}: {value}"

        prompt += "\n\nStructurez le rapport avec des titres et des listes."
        return self.provider.generate_text(prompt)

    def generate_summary(self, text: str) -> str:
        prompt = f"Résumez le texte suivant en 3 points clés :\n{text}"
        return self.provider.generate_text(prompt)

In [ ]:
# Test avec Ollama (fonctionne sans clé API)
donnees = {
    "Produit": "Application mobile d'apprentissage",
    "Catégorie": "Éducation",
    "Nombre d'utilisateurs": "10 000",
    "Taux de satisfaction": "4.5/5"
}

report_generator = ReportGenerator(ollama_provider)
report = report_generator.generate_report(donnees)
print("=== Rapport ===")
print(report)

print("\n=== Résumé ===")
print(report_generator.generate_summary(report))

## Exemple concret : assistant conversationnel

Créons maintenant un **chatbot** qui garde en mémoire le fil de la conversation. Ici encore, la classe est indépendante du fournisseur :

In [ ]:
class ChatAssistant:
    """Assistant conversationnel avec mémoire, basé sur n'importe quel AIProvider."""

    def __init__(self, provider: AIProvider, system_prompt: str = ""):
        self.provider = provider
        self.system_prompt = system_prompt
        self.history = []  # liste de tuples (message utilisateur, réponse)

    def send(self, message: str) -> str:
        # On reconstruit tout le contexte : instruction système + historique
        prompt = ""
        if self.system_prompt:
            prompt += f"[Système] {self.system_prompt}\n"
        for user_msg, bot_msg in self.history:
            prompt += f"[Utilisateur] {user_msg}\n[Assistant] {bot_msg}\n"
        prompt += f"[Utilisateur] {message}"

        response = self.provider.generate_text(prompt)
        self.history.append((message, response))
        return response

In [ ]:
# Création d'un assistant qui parle français et reste concis
assistant = ChatAssistant(
    ollama_provider,
    system_prompt="Tu es un assistant pédagogique en Python. Réponds de façon courte en français."
)

print("Assistant :", assistant.send("Qu'est-ce qu'une liste en Python ?"))
print()
print("Assistant :", assistant.send("Et comment y ajouter un élément ?"))

## Comparaison et choix du fournisseur

| Critère | Google (Gemini) | Ollama (local) | OpenRouter | Groq |
|---------|-----------------|----------------|------------|------|
| Coût | Gratuit (free tier) | Gratuit | Gratuit (`:free`) | Gratuit |
| Carte bancaire | Non | Non | Non | Non |
| Connexion internet | Requise | Non (hors ligne) | Requise | Requise |
| Confidentialité | Données envoyées à Google | 100 % locale | Données envoyées à OpenRouter | Données envoyées à Groq |
| Puissance du matériel | Peu importe | Modèles ≤ 3B recommandés | Peu importe | Peu importe |
| Vitesse | Moyenne | Selon le matériel | Moyenne | Très rapide |
| Qualité des réponses | Très bonne (Gemini 2.5 Flash) | Correcte (petits modèles) | Variable selon le modèle | Bonne (Llama 3.1 8B) |
| Limites gratuites | 10-15 req/min | Aucune | 20 req/min, 50 req/jour | 30 req/min, 14 400 req/jour |

## Bonnes pratiques

1. **Choisir le fournisseur selon le contexte**
   - Machine modeste ou pas d'internet → **Ollama** avec un petit modèle
   - Besoin de qualité sans configurer de matériel → **Google** (free tier)
   - Tester plusieurs modèles avec une seule clé → **OpenRouter**
   - Besoin de rapidité en temps réel → **Groq**

2. **Gestion des erreurs**
   - Toujours gérer les exceptions réseau ou API (elles sont déjà gérées dans nos classes)
   - Vérifier que Ollama est démarré : `ollama serve` (ou relancer l'application)
   - Respecter les limites de requêtes (attendre quelques secondes entre deux appels si limité)

3. **Sécurité**
   - Ne jamais écrire une clé API en dur dans le code
   - Utiliser des variables d'environnement ou un fichier `.env` (ajouté au `.gitignore`)
   - Ne pas envoyer de données sensibles à des API cloud si la confidentialité est critique

4. **Qualité des résultats**
   - Vérifier que les réponses sont pertinentes (les modèles peuvent se tromper : hallucinations)
   - Adapter le prompt (instructions claires et précises) pour de meilleurs résultats

## Conclusion

L'intégration de l'IA avec Python est accessible à tous, **même avec un petit budget et un petit ordinateur** :

- **Google Generative AI** : des modèles puissants gratuitement, sans carte bancaire (free tier sur les modèles Flash)
- **Ollama** : des modèles locaux gratuits à 100 %, parfaits sur du matériel modeste avec des modèles de 0,5 à 3 milliards de paramètres
- **OpenRouter** : un accès unique à de nombreux modèles gratuits `:free`
- **Groq** : une API gratuite et ultra-rapide en bonus

### Points clés à retenir

- Utiliser une **classe abstraite** (`AIProvider`) pour uniformiser les interfaces et changer de fournisseur sans réécrire le code
- Vérifier les **limites gratuites** et les **tarifs**, qui changent régulièrement
- Adapter le **choix du modèle** à son matériel et à ses besoins
- Ne jamais **partager ses clés API** ni les écrire en dur

### Pour aller plus loin

- **API compatible OpenAI** : Groq et OpenRouter utilisent le même format, le module `openai` sert aux deux
- **Streaming** : afficher la réponse token par token (`stream=True`)
- **Modèles multimodaux** : Gemini et certains modèles OpenRouter acceptent aussi des images en entrée